In [1]:
"""
Tutorial: TF perturbation inference from WT and motif-masked KO inputs

This notebook demonstrates the full workflow:
1. Load DEG gene lists for one TF (up and down separately)
2. Build WT inputs for the selected genes
3. Scan TF motif and generate KO inputs by masking matched peaks
4. Convert WT and KO txt inputs into model-ready tensors
5. Run inference and compute predicted expression delta
6. Evaluate direction accuracy, macro-F1, and PCC with log2FC
"""

'\nTutorial: TF perturbation inference from WT and motif-masked KO inputs\n\nThis notebook demonstrates the full workflow:\n1. Load DEG gene lists for one TF (up and down separately)\n2. Build WT inputs for the selected genes\n3. Scan TF motif and generate KO inputs by masking matched peaks\n4. Convert WT and KO txt inputs into model-ready tensors\n5. Run inference and compute predicted expression delta\n6. Evaluate direction accuracy, macro-F1, and PCC with log2FC\n'

In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd

from models import *

from toolkit import (
    set_seed,
    parse_tf_deg_filename,
    build_tutorial_gene_tables,
    collect_union_gene_list,
    prepare_tutorial_outputs,
    GeneATACPreprocessor,
    FIMOMotifScanner,
    build_wild_type_inputs_for_genes,
    TFKOPeakMasker,
    rebuild_wt_meta_from_txt_and_gtf,
    load_wt_meta,
    build_gene_samples_from_txt_dir,
    build_ko_gene_to_sample_map_from_dir,
    load_model_components,
    run_delta_inference,
    evaluate_prediction_table,
    print_evaluation_summary,
    save_results_table,
    save_metrics_json,
)

from data_processing import (
    narrowPeak_Reader,
    tokenizer,
    pre_processing,
    pre_pro,
)


In [3]:
# =========================
# User configuration
# =========================
SEED = 42
set_seed(SEED)

CUDA_DEVICE = 0
DEVICE = torch.device(f"cuda:{CUDA_DEVICE}" if torch.cuda.is_available() else "cpu")

# -------------------------
# Input files
# -------------------------
GTF_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/genes-2020-A.gtf"
GENOME_DIR = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/genome"
PEAK_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/ENCFF333TAT.narrowPeak"
BIGWIG_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/ENCFF357GNC.bigWig"

MOTIF_DIR = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/motifs"
FIMO_PATH = "fimo"

UP_CSV_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/K562_GATA1_up.csv"
DOWN_CSV_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/K562_GATA1_down.csv"

MODEL_CHECKPOINT_DIR = "/mnt/nas-data/yzk/pertformer-tutorials/checkpoints"
GENOME_DICT_PATH = "/mnt/nas-data/yzk/pertformer-tutorials/example_data/kmer_dict.pkl"

# -------------------------
# Output root
# -------------------------
OUTPUT_ROOT = "/mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output"

# -------------------------
# Sampling options
# Set to None for full set
# -------------------------
MAX_UP_GENES = 50
MAX_DOWN_GENES = 50

# -------------------------
# Runtime options
# -------------------------
VERBOSE_EVERY = 10
STRONG_PVALUE_THRESH = 1e-5
FALLBACK_PVALUE_THRESH = 1e-4


In [4]:
parsed_up = parse_tf_deg_filename(UP_CSV_PATH)
parsed_down = parse_tf_deg_filename(DOWN_CSV_PATH)

if parsed_up is None or parsed_down is None:
    raise ValueError("Failed to parse input CSV filenames.")

cell_line_up, tf_name_up, direction_up = parsed_up
cell_line_down, tf_name_down, direction_down = parsed_down

if cell_line_up != cell_line_down or tf_name_up != tf_name_down:
    raise ValueError("The up/down CSV files must belong to the same TF and cell line.")

if direction_up != "up" or direction_down != "down":
    raise ValueError("Expected one *_up.csv and one *_down.csv file.")

CELL_LINE = cell_line_up
TF_NAME = tf_name_up

PATHS = prepare_tutorial_outputs(
    output_root=OUTPUT_ROOT,
    tf_name=TF_NAME,
    cell_line=CELL_LINE
)

print("Configuration summary")
print("---------------------")
print(f"Cell line         : {CELL_LINE}")
print(f"TF name           : {TF_NAME}")
print(f"Up CSV            : {UP_CSV_PATH}")
print(f"Down CSV          : {DOWN_CSV_PATH}")
print(f"Output root       : {OUTPUT_ROOT}")
print(f"WT dir            : {PATHS['wt_dir']}")
print(f"KO root           : {PATHS['ko_root']}")
print(f"Result CSV        : {PATHS['result_csv']}")
print(f"Metric JSON       : {PATHS['metric_json']}")
print(f"Device            : {DEVICE}")
print("")


Configuration summary
---------------------
Cell line         : K562
TF name           : GATA1
Up CSV            : /mnt/nas-data/yzk/pertformer-tutorials/example_data/K562_GATA1_up.csv
Down CSV          : /mnt/nas-data/yzk/pertformer-tutorials/example_data/K562_GATA1_down.csv
Output root       : /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output
WT dir            : /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/wild_type_txt
KO root           : /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt
Result CSV        : /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/results/K562_GATA1_tutorial_predictions.csv
Metric JSON       : /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/results/K562_GATA1_tutorial_metrics.json
Device            : cuda:0



In [5]:
up_df, down_df, merged_df = build_tutorial_gene_tables(
    up_csv_path=UP_CSV_PATH,
    down_csv_path=DOWN_CSV_PATH,
    max_up=MAX_UP_GENES,
    max_down=MAX_DOWN_GENES,
    random_seed=SEED
)

gene_list = collect_union_gene_list(up_df, down_df)

print("Selected DEG summary")
print("--------------------")
print(f"Sampled up genes    : {len(up_df)}")
print(f"Sampled down genes  : {len(down_df)}")
print(f"Union genes         : {len(gene_list)}")
print("")

display(merged_df.head())


Selected DEG summary
--------------------
Sampled up genes    : 50
Sampled down genes  : 50
Union genes         : 100



,cell_line,perturbed_tf,Gene,log2FC,padj,selected_method,all_methods_detected,all_directions_detected,direction_agree,expected_direction
0,K562,GATA1,LGALS3BP,2.927241,4.207270e-13,CRISPRi-RNA-seq,CRISPRi-RNA-seq,CRISPRi-RNA-seq:up,True,up
1,K562,GATA1,NEU1,2.580279,1.111400e-12,CRISPRi-RNA-seq,CRISPRi-RNA-seq,CRISPRi-RNA-seq:up,True,up
2,K562,GATA1,BAIAP3,2.880596,3.329510e-11,CRISPRi-RNA-seq,CRISPRi-RNA-seq,CRISPRi-RNA-seq:up,True,up
3,K562,GATA1,HMGCL,2.413678,4.713690e-11,CRISPRi-RNA-seq,CRISPRi-RNA-seq,CRISPRi-RNA-seq:up,True,up
4,K562,GATA1,CRIP2,4.527539,9.332760e-11,CRISPRi-RNA-seq,CRISPRi-RNA-seq,CRISPRi-RNA-seq:up,True,up


In [6]:
print("Initializing preprocessing resources...")
preprocessor = GeneATACPreprocessor(
    gtf_path=GTF_PATH,
    genome_dir=GENOME_DIR,
    peak_path=PEAK_PATH,
    bigwig_path=BIGWIG_PATH
)

motif_scanner = FIMOMotifScanner(
    motif_dir=MOTIF_DIR,
    strong_pvalue_thresh=STRONG_PVALUE_THRESH,
    fallback_pvalue_thresh=FALLBACK_PVALUE_THRESH,
    fimo_path=FIMO_PATH
)

print("Preprocessing resources are ready.\n")

Initializing preprocessing resources...
[1/4] Loading GTF index...
  Loaded 36591 genes from GTF.
[2/4] Loading chromosome tensors...


/home/yangzikun.yzk/tutorials/Tutorials_TF_knockdown/toolkit.py:127: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.genome[i] = torch.load(f"{genome_dir}/chr{i+1}.pkl")


  Genome tensors loaded.
[3/4] Loading and deduplicating peaks...
  Loaded 161693 unique peaks.
[4/4] Opening bigWig...
  bigWig opened successfully.

Preprocessing resources are ready.



In [7]:
wt_meta_path = build_wild_type_inputs_for_genes(
    preprocessor=preprocessor,
    gene_list=gene_list,
    wild_type_dir=PATHS["wt_dir"],
    verbose_every=VERBOSE_EVERY,
    suppress_inner_print=True
)

print("WT input generation done.")
print(f"WT meta path: {wt_meta_path}")

Building WT inputs for 100 genes...
  WT progress: 1/100 | success=1 failed=0 existing=0
  WT progress: 10/100 | success=9 failed=1 existing=0
  WT progress: 20/100 | success=19 failed=1 existing=0
  WT progress: 30/100 | success=29 failed=1 existing=0
  WT progress: 40/100 | success=37 failed=3 existing=0
  WT progress: 50/100 | success=47 failed=3 existing=0
  WT progress: 60/100 | success=57 failed=3 existing=0
  WT progress: 70/100 | success=67 failed=3 existing=0
  WT progress: 80/100 | success=77 failed=3 existing=0
  WT progress: 90/100 | success=87 failed=3 existing=0
  WT progress: 100/100 | success=97 failed=3 existing=0
WT generation finished. success=97, failed=3, existing=0
WT meta saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/wild_type_txt/meta_info.json
WT input generation done.
WT meta path: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/wild_type_txt/meta_info.json


In [8]:
print("Rebuilding WT metadata from generated txt files...")
wt_meta_path = rebuild_wt_meta_from_txt_and_gtf(
    wild_type_dir=PATHS["wt_dir"],
    preprocessor=preprocessor,
    output_meta_path=PATHS["wt_meta_path"]
)

print(f"Usable WT meta saved to: {wt_meta_path}\n")

Rebuilding WT metadata from generated txt files...
Usable WT meta saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/wild_type_txt/meta_info.json



In [9]:
ko_masker = TFKOPeakMasker(
    preprocessor=preprocessor,
    wild_type_dir=PATHS["wt_dir"],
    ko_base_dir=PATHS["ko_root"],
    motif_scanner=motif_scanner
)

up_gene_to_deg = {row["Gene"]: row for _, row in up_df.iterrows()}

ko_up_dir, ko_up_meta = ko_masker.process_gene_list_for_tf(
    tf_name=TF_NAME,
    cell_line=CELL_LINE,
    deg_direction="up",
    gene_to_deg_info=up_gene_to_deg,
    verbose_every=VERBOSE_EVERY
)

print(f"KO up directory: {ko_up_dir}")


Generating KO inputs: K562_GATA1_up
  Number of genes: 50
  KO progress: 1/50 | success=0 skipped=0
  KO progress: 10/50 | success=7 skipped=2
  KO progress: 20/50 | success=16 skipped=3
  KO progress: 30/50 | success=23 skipped=6
  KO progress: 40/50 | success=31 skipped=8
  KO progress: 50/50 | success=41 skipped=8
KO generation finished: success=42, skipped=8
KO meta saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_up/ko_meta_info.json

KO up directory: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_up


In [10]:
down_gene_to_deg = {row["Gene"]: row for _, row in down_df.iterrows()}

ko_down_dir, ko_down_meta = ko_masker.process_gene_list_for_tf(
    tf_name=TF_NAME,
    cell_line=CELL_LINE,
    deg_direction="down",
    gene_to_deg_info=down_gene_to_deg,
    verbose_every=VERBOSE_EVERY
)

print(f"KO down directory: {ko_down_dir}")


Generating KO inputs: K562_GATA1_down
  Number of genes: 50
  KO progress: 1/50 | success=0 skipped=0
  KO progress: 10/50 | success=9 skipped=0
  KO progress: 20/50 | success=19 skipped=0
  KO progress: 30/50 | success=27 skipped=2
  KO progress: 40/50 | success=34 skipped=5
  KO progress: 50/50 | success=41 skipped=8
KO generation finished: success=42, skipped=8
KO meta saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_down/ko_meta_info.json

KO down directory: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_down


In [11]:
print("Loading genome dictionary and WT metadata...")
genome_dict = torch.load(GENOME_DICT_PATH)
gene_meta = load_wt_meta(wt_meta_path)

print(f"Loaded WT meta for {len(gene_meta)} genes.\n")

Loading genome dictionary and WT metadata...
Loaded WT meta for 97 genes.



/tmp/ipykernel_348223/1045301463.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  genome_dict = torch.load(GENOME_DICT_PATH)


In [12]:
wt_gene_to_sample = build_gene_samples_from_txt_dir(
    txt_dir=PATHS["wt_dir"],
    gene_list=gene_list,
    gene_meta=gene_meta,
    narrowPeak_Reader=narrowPeak_Reader,
    tokenizer=tokenizer,
    pre_processing=pre_processing,
    pre_pro=pre_pro,
    genome_dict=genome_dict,
    verbose_every=VERBOSE_EVERY
)

print(f"WT tensor-ready genes: {len(wt_gene_to_sample)}")


Preparing model-ready tensors from: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/wild_type_txt
  Candidate genes with meta: 97
  Tensor prep progress: 1/97 | success=1 skipped=0
  Tensor prep progress: 10/97 | success=10 skipped=0
  Tensor prep progress: 20/97 | success=20 skipped=0
  Tensor prep progress: 30/97 | success=30 skipped=0
  Tensor prep progress: 40/97 | success=40 skipped=0
  Tensor prep progress: 50/97 | success=50 skipped=0
  Tensor prep progress: 60/97 | success=60 skipped=0
  Tensor prep progress: 70/97 | success=70 skipped=0
  Tensor prep progress: 80/97 | success=80 skipped=0
  Tensor prep progress: 90/97 | success=90 skipped=0
  Tensor prep progress: 97/97 | success=97 skipped=0
Tensor preparation finished. success=97, skipped=0

WT tensor-ready genes: 97


In [13]:
ko_up_gene_to_sample = build_ko_gene_to_sample_map_from_dir(
    ko_dir=ko_up_dir,
    gene_list=up_df["Gene"].tolist(),
    gene_meta=gene_meta,
    narrowPeak_Reader=narrowPeak_Reader,
    tokenizer=tokenizer,
    pre_processing=pre_processing,
    pre_pro=pre_pro,
    genome_dict=genome_dict,
    verbose_every=VERBOSE_EVERY
)

print(f"KO-up tensor-ready genes: {len(ko_up_gene_to_sample)}")


Preparing model-ready tensors from: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_up
  Candidate genes with meta: 48
  Tensor prep progress: 10/48 | success=8 skipped=2
  Tensor prep progress: 20/48 | success=18 skipped=2
  Tensor prep progress: 30/48 | success=26 skipped=4
  Tensor prep progress: 40/48 | success=34 skipped=6
  Tensor prep progress: 48/48 | success=42 skipped=6
Tensor preparation finished. success=42, skipped=6

KO-up tensor-ready genes: 42


In [14]:
ko_down_gene_to_sample = build_ko_gene_to_sample_map_from_dir(
    ko_dir=ko_down_dir,
    gene_list=down_df["Gene"].tolist(),
    gene_meta=gene_meta,
    narrowPeak_Reader=narrowPeak_Reader,
    tokenizer=tokenizer,
    pre_processing=pre_processing,
    pre_pro=pre_pro,
    genome_dict=genome_dict,
    verbose_every=VERBOSE_EVERY
)

print(f"KO-down tensor-ready genes: {len(ko_down_gene_to_sample)}")


Preparing model-ready tensors from: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ko_txt/K562_GATA1_down
  Candidate genes with meta: 49
  Tensor prep progress: 1/49 | success=1 skipped=0
  Tensor prep progress: 10/49 | success=10 skipped=0
  Tensor prep progress: 49/49 | success=42 skipped=7
Tensor preparation finished. success=42, skipped=7

KO-down tensor-ready genes: 42


In [15]:
model_parts = load_model_components(
    checkpoint_dir=MODEL_CHECKPOINT_DIR,
    device=DEVICE
)

Loading model components...
Model components loaded.



In [16]:
result_df = run_delta_inference(
    genes_with_labels=merged_df[["Gene", "expected_direction", "log2FC"]],
    wt_gene_to_sample=wt_gene_to_sample,
    ko_up_gene_to_sample=ko_up_gene_to_sample,
    ko_down_gene_to_sample=ko_down_gene_to_sample,
    model_parts=model_parts,
    device=DEVICE,
    verbose_every=VERBOSE_EVERY
)

print("Prediction table preview")
print("------------------------")
display(result_df.head())

Running inference for 100 genes...
  Inference progress: 1/100 | success=0
  Inference progress: 10/100 | success=7
  Inference progress: 20/100 | success=17
  Inference progress: 30/100 | success=24
  Inference progress: 40/100 | success=32
  Inference progress: 50/100 | success=42
  Inference progress: 60/100 | success=52
  Inference progress: 70/100 | success=61
  Inference progress: 80/100 | success=69
  Inference progress: 90/100 | success=77
  Inference progress: 100/100 | success=84
Inference finished.

Prediction table preview
------------------------


,Gene,expected_direction,log2FC,pred_delta,pred_direction,status
0,LGALS3BP,up,2.927241,NaN,None,missing_sample
1,NEU1,up,2.580279,8.125000,up,success
2,BAIAP3,up,2.880596,8.198242,up,success
3,HMGCL,up,2.413678,NaN,None,missing_sample
4,CRIP2,up,4.527539,7.191406,up,success


In [17]:
metrics = evaluate_prediction_table(result_df)
print_evaluation_summary(metrics)

Evaluation summary
------------------
Valid genes            : 84
Up accuracy            : 0.9524
Down accuracy          : 0.8095
Overall accuracy       : 0.8810
Macro-F1               : 0.8803
PCC(pred_delta, log2FC): 0.6635

